# 10 — Gate AB-4: the ensemble runner (R)

Kernel `R (y2y)`; live internet. Mirrors the parent's `12_gate4_ensemble` + `18_guarded_sweep`
in one pass: for every frozen formulation, at the primary budget level, into
`runs/ab_l/<level>/<formulation_id>/`:

| artifact | what | config |
|---|---|---|
| `anchor/` | engine certified anchor | Gurobi binary, opt_gap 1e-4, NumericFocus |
| `twin/` | LP twin | Gurobi proportion |
| `kbest/` | k-best pool (the two-instrument contrast, E5) | Gurobi binary, portfolio 50 @ 5% |
| `anchor.tif` + `formulation_meta.json` | MGA anchor, drift-checked vs `anchor/` | `mga_core` |
| `mga_g05.tif` | aggregate 5% band, 50 members | `mga_maxham_v1` |
| `mga_guard_g05.tif` | guarded band (per-block floors 0.95) | `mga_block_floors` |
| `mga_g02.tif` + `mga_guard_g02.tif` | **the applied band (D-AB10, g = 2%)**, both semantics — the per-scenario frequency products | same machinery |

Fully resumable per artifact (the reference formulation's AB-1/AB-2 artifacts are reused as-is).
Verifies `manifest.csv` against its freeze hash before solving. At AB scale expect ~2–5 min per
formulation; budget an hour.

In [ ]:
# ---- setup + freeze verification ------------------------------------------------------------------------
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))
HERE <- file.path(PROJ, "analyses", "alberta_prioritization")
mpath <- file.path(PROJ, "input_data", "aligned_stack_ab", "manifest.json")
stopifnot(file.exists(mpath))
MAN <- read.csv(file.path(HERE, "spec", "manifest.csv"), stringsAsFactors = FALSE)
stopifnot(nrow(MAN) == 14)
dig <- strsplit(readLines(file.path(HERE, "spec", "manifest_freeze.sha256"))[1], "  ")[[1]][1]
stopifnot("manifest.csv does not match its freeze hash -- STOP" =
            identical(unname(tools::sha256sum(file.path(HERE, "spec", "manifest.csv"))[[1]]), dig))
cat(sprintf("manifest verified against freeze hash %s...\n", substr(dig, 1, 16)))
LEVEL <- unique(MAN$budget_level); stopifnot(length(LEVEL) == 1)
BUDGET_PCT <- unique(MAN$budget_pct); stopifnot(length(BUDGET_PCT) == 1)
SC <- jsonlite::read_json(file.path(HERE, "spec", "scenarios_ab_v1.json"))
BLOCKS <- lapply(SC$`_meta`$blocks, unlist)
RUNS_REL <- file.path("analyses/alberta_prioritization/runs/ab_l", LEVEL)
REAL245 <- "input_data/aligned_stack_ab/climate_realizations/macrorefugia_245_2071_2100.tif"
FLOOR_G <- unique(MAN$floor_g); stopifnot(length(FLOOR_G) == 1)
APPLIED_G <- unique(MAN$applied_band_g); stopifnot(length(APPLIED_G) == 1)   # D-AB10: g = 2%
cat(sprintf("level %s | budget_pct %.4f | floors on %s (g %.2f)\n", LEVEL, BUDGET_PCT, paste(names(BLOCKS), collapse = ", "), FLOOR_G))

ctx585 <- pr_setup(mpath, PROJ); ctx585 <- modifyList(ctx585, pr_ingest(ctx585))
ctx245 <- pr_setup(mpath, PROJ)
ctx245$layers$path[ctx245$layers$name == "climate_type_macrorefugia"] <- REAL245
ctx245 <- modifyList(ctx245, pr_ingest(ctx245))
base_for <- function(row) {
  b <- if (grepl("^ssp245", row$climate_level)) ctx245 else ctx585
  b <- pr_override(b, budget_pct = BUDGET_PCT, results_dir = file.path(RUNS_REL, row$formulation_id), results_subdir = "_base")
  modifyList(b, pr_planning_units(b))
}
form_wt <- function(row) list(w = jsonlite::fromJSON(row$weight_vector), t = jsonlite::fromJSON(row$target_vector))

In [ ]:
# ---- DRY PLAN (no solves) ---------------------------------------------------------------------------------
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]; cd <- file.path(PROJ, RUNS_REL, row$formulation_id)
  st <- function(f) if (file.exists(file.path(cd, f))) "done" else "TODO"
  cat(sprintf("%-22s %-7s anchor:%-5s twin:%-5s kbest:%-5s mga05:%-5s guard05:%-5s mga02:%-5s guard02:%-5s\n", row$formulation_id,
              sub("_2071_2100", "", row$climate_level), st("anchor/run_summary.json"), st("twin/run_summary.json"),
              st("kbest/run_summary.json"), st("mga_g05.tif"), st("mga_guard_g05.tif"), st("mga_g02.tif"), st("mga_guard_g02.tif")))
}

In [ ]:
# ---- runners -------------------------------------------------------------------------------------------------
run_engine <- function(row, artifact, ov) {
  done <- file.path(PROJ, RUNS_REL, row$formulation_id, artifact, "run_summary.json")
  if (file.exists(done)) { cat(sprintf("   %s/%s exists -- skipped\n", row$formulation_id, artifact)); return(invisible(NULL)) }
  wt <- form_wt(row)
  actx <- do.call(pr_override, c(list(base_for(row), targets = wt$t, feature_weight_multipliers = wt$w,
                                      results_subdir = artifact), ov))
  actx <- modifyList(actx, pr_weights(actx)); actx <- modifyList(actx, pr_targets(actx)); actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  sv <- pr_solve(actx); actx$s <- sv$s; actx$timing <- sv$timing; actx$n_sol <- sv$n_sol; actx$sol_attrs <- sv$sol_attrs
  actx <- modifyList(actx, pr_summaries(actx))
  pr_write_outputs(actx)
  invisible(NULL)
}
run_mga <- function(row) {
  cd <- file.path(PROJ, RUNS_REL, row$formulation_id)
  if (all(file.exists(file.path(cd, c("mga_g05.tif", "mga_guard_g05.tif", "mga_g02.tif", "mga_guard_g02.tif"))))) {
    cat(sprintf("   %s/mga + guard exist -- skipped\n", row$formulation_id)); return(invisible(NULL)) }
  eng <- file.path(cd, "anchor", "run_summary.json"); stopifnot(file.exists(eng))
  z_eng <- as.numeric(unlist(jsonlite::read_json(eng)$solver_provenance$objective))[1]
  wt <- form_wt(row)
  actx <- pr_override(base_for(row), targets = wt$t, feature_weight_multipliers = wt$w, results_subdir = "mga_build",
                      solver = "gurobi", decision_type = "binary", opt_gap = row$opt_gap, portfolio_n = 1)
  actx <- modifyList(actx, pr_weights(actx)); actx <- modifyList(actx, pr_targets(actx)); actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  cm <- mga_compile(actx)
  anchor <- mga_anchor(cm, opt_gap = row$opt_gap)
  rel <- abs(anchor$z - z_eng) / abs(z_eng)
  stopifnot("MGA anchor drifted > 1e-3 from the engine certificate -- STOP" = rel <= 1e-3)
  if (!file.exists(file.path(cd, "anchor.tif"))) {
    r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r)); v[cm$pu_index] <- as.integer(anchor$x)
    terra::values(r) <- v
    terra::writeRaster(r, file.path(cd, "anchor.tif"), overwrite = TRUE, datatype = "INT1U", NAflag = 255,
                       gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
  }
  if (!file.exists(file.path(cd, "formulation_meta.json")))
    jsonlite::write_json(list(formulation_id = row$formulation_id, level = LEVEL, estimator = row$estimator,
                              anchor_objective = anchor$z, anchor_bound = anchor$bound, anchor_gap = anchor$gap,
                              anchor_runtime_s = anchor$runtime, engine_anchor_objective = z_eng, anchor_rel_drift = rel,
                              k = row$k_requested, g = row$band_gap_g, floor_g = FLOOR_G, blocks = BLOCKS,
                              opt_gap = row$opt_gap, mip_gap_dist = 0.01, time_limit_iter = 900,
                              created_utc = format(Sys.time(), tz = "UTC")),
                         file.path(cd, "formulation_meta.json"), auto_unbox = TRUE, pretty = TRUE, digits = 10)
  if (!file.exists(file.path(cd, "mga_g05.tif"))) {
    gen <- mga_generate(cm, anchor, g = row$band_gap_g, k = row$k_requested); mga_write(gen, cm, actx$cost, cd, "g05") }
  if (!file.exists(file.path(cd, "mga_guard_g05.tif"))) {
    gen <- mga_generate(cm, anchor, g = row$band_gap_g, k = row$k_requested,
                        floors = list(ctx = actx, blocks = BLOCKS, g = FLOOR_G))
    mga_write(gen, cm, actx$cost, cd, "guard_g05") }
  # D-AB10: the applied band -- both semantics, every formulation (the per-scenario frequency products)
  if (!file.exists(file.path(cd, "mga_g02.tif"))) {
    gen <- mga_generate(cm, anchor, g = APPLIED_G, k = row$k_requested); mga_write(gen, cm, actx$cost, cd, "g02") }
  if (!file.exists(file.path(cd, "mga_guard_g02.tif"))) {
    gen <- mga_generate(cm, anchor, g = APPLIED_G, k = row$k_requested,
                        floors = list(ctx = actx, blocks = BLOCKS, g = FLOOR_G))
    mga_write(gen, cm, actx$cost, cd, "guard_g02") }
  invisible(NULL)
}

In [ ]:
# ---- THE LOOP (serial, resumable anywhere) --------------------------------------------------------------
t_batch <- proc.time()[["elapsed"]]
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]
  cat(sprintf("\n===================== %s (%d/%d) =====================\n", row$formulation_id, i, nrow(MAN)))
  run_engine(row, "anchor", list(solver = "gurobi", decision_type = "binary", opt_gap = row$opt_gap, portfolio_n = 1))
  run_engine(row, "twin",   list(solver = "gurobi", decision_type = "proportion", opt_gap = row$opt_gap, portfolio_n = 1))
  run_engine(row, "kbest",  list(solver = "gurobi", decision_type = "binary", opt_gap = row$opt_gap,
                                 portfolio_n = row$k_requested, portfolio_gap = row$band_gap_g))
  run_mga(row)
  cat(sprintf("== %s done | batch elapsed %.1f min\n", row$formulation_id, (proc.time()[["elapsed"]] - t_batch) / 60))
}
cat("\nAB-4 ENSEMBLE COMPLETE -- next: 11_ab4_analysis.ipynb (kernel y2y-geo)\n")

In [ ]:
# ---- integrity summary ------------------------------------------------------------------------------------
obj_of <- function(p) tryCatch(as.numeric(unlist(jsonlite::read_json(p)$solver_provenance$objective))[1], error = function(e) NA)
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]; cd <- file.path(PROJ, RUNS_REL, row$formulation_id)
  fm <- file.path(cd, "formulation_meta.json"); if (!file.exists(fm)) { cat(sprintf("%-22s INCOMPLETE\n", row$formulation_id)); next }
  m <- jsonlite::read_json(fm); tw <- obj_of(file.path(cd, "twin", "run_summary.json"))
  ce <- read.csv(file.path(cd, "certificates_g05.csv")); cg <- read.csv(file.path(cd, "certificates_guard_g05.csv"))
  c2 <- read.csv(file.path(cd, "certificates_g02.csv")); c2g <- read.csv(file.path(cd, "certificates_guard_g02.csv"))
  kb <- jsonlite::read_json(file.path(cd, "kbest", "run_summary.json"))
  cat(sprintf("%-22s anchor %.6f (%3.0fs, drift %.1e) | twin %.6f [LP<=MILP %s] | kbest %2d | g05 %2d/%s guard %2d/%s | g02 %2d/%s guard %2d/%s | %.1f min\n",
              row$formulation_id, m$anchor_objective, m$anchor_runtime_s, m$anchor_rel_drift, tw,
              ifelse(tw <= m$anchor_objective + 1e-6, "OK", "VIOLATED"), kb$n_alternatives,
              nrow(ce), if (all(ce$band_ok)) "OK" else "VIOL", nrow(cg), if (all(cg$band_ok)) "OK" else "VIOL",
              nrow(c2), if (all(c2$band_ok)) "OK" else "VIOL", nrow(c2g), if (all(c2g$band_ok)) "OK" else "VIOL",
              (sum(ce$runtime_s) + sum(cg$runtime_s) + sum(c2$runtime_s) + sum(c2g$runtime_s)) / 60))
}